### feature = 문제 / 입력 데이터 / 모델이 보는 정보
### target  = 정답 / 예측해야 하는 값

## 하이퍼파라미터 기본 개념
하이퍼파라미터: 모델이 데이터에서 스스로 학습하는 값이 아니라, 사람이 학습 전에 미리 정하는 설정값임.

- alpha: 규제를 얼마나 강하게 적용할지 정하는 값임.
- l1_ratio: ElasticNet에서 L1 규제와 L2 규제를 어떤 비율로 섞을지 정하는 값임.

# Regularized Linear Models 규제 선형 회귀
다항식이 복잡해지져서 회귀계수가 매우 크게 설정이되면서 과대적합이 되고
평가데이터세트에 대해서 형편없는 예측 성능을 보이게 된다.

비용함수의 최소값을 구하는 것이 모델이 추구하는 목적이므로,
비용함수의 최소값을 구하는데 **𝝰 제약**을 걸어 과적합을 방지하는 것을 규제라고 한다.
- Lasso L1방식의 규제 적용
- Ridge L2방식의 규제 적용
- ElasticNet L1, L2규제를 결합한 모델. 특성이 많은 데이터셋에 적용. L1규제로 특성개수를 줄이고, L2규제로 계수값의 크기도 조정할 수 있다.


**비용함수 목표**

$
비용함수 목표 = \min \left(\text{RSS}(w) + \alpha \times W\right)
$

이 수식은 규제를 적용한 비용함수를 의미하며, 다음과 같은 요소들로 이루어져 있다:
1. **RSS(w)**: 이 부분은 Residual Sum of Squares의 약자로, 잔차 제곱합을 의미한다. 선형 회귀 모델에서 주로 사용하는 손실 함수로, 각 데이터 포인트에서의 예측 값과 실제 값 간의 차이를 제곱한 것들의 합이다. 즉, 모델의 예측 오차를 측정하는 부분이다. 수식으로는 다음과 같이 표현된다:
    $
    \text{RSS}(w) = \sum_{i=1}^n \left(y_i - \hat{y}_i\right)^2
    $
   여기서 $y_i$는 실제 값, $\hat{y}_i$는 모델의 예측 값이다.
2. **$\alpha$**: 규제 강도를 나타내는 하이퍼파라미터이다. 이 값이 클수록 규제의 효과가 커지고, 작을수록 규제의 효과가 줄어든다. 모델이 과적합되기 쉬운 경우, $\alpha$를 크게 설정하여 가중치를 제어할 수 있다.
3. **$W$**: 가중치들의 규제 항을 의미한다. 이는 가중치의 크기에 페널티를 부여하는 부분으로, 모델의 복잡도를 조절하는 역할을 한다.
   - L1 규제: $W = \sum_{j=1}^p |w_j|$
   - L2 규제: $W = \sum_{j=1}^p w_j^2$
수식을 다시 설명하면, 비용함수의 목표는 잔차 제곱합(RSS)과 규제 항($\alpha \times W$)을 더한 값을 최소화하는 것이다. 즉, 이 비용함수의 최적화 목표는 두 가지를 달성하고자 한다:
1. 모델의 예측 오차(RSS)를 줄이는 것.
2. 모델의 복잡도를 줄여서 가중치의 크기를 제어하는 것($\alpha \times W$).

**적합합 규제를 선택하려면 :**

1. **Lasso (L1 규제)**
  - 불필요한 피처를 자동으로 제거할 때 유용하다.
  - 많은 피처 중 일부만 중요할 때 사용하면 스파스한 모델을 만듦.
2. **Ridge (L2 규제)**
  - 모든 피처가 유의미하고 예측에 기여한다고 생각될 때 적합하다.
  - 피처가 많고 과적합을 방지하고 싶을 때 사용한다.
3. **Elastic Net (L1 + L2 규제)**
  - 피처를 일부 제거하면서도 나머지의 가중치도 줄이고 싶을 때 유용하다.
  - 상관관계가 높은 피처가 있을 때 선택을 안정적으로 한다.


## L2
- L2방식의 규제를 구현한 Ridge 클래스를 사용할 수 있다.
- 모든 피쳐의 회귀계수를 규제해 과적합을 방지한다.


## 실습 환경 준비

- NumPy: 배열 계산과 수치 연산을 다루기 위한 기본 라이브러리임.
- Pandas: 표 형태 데이터를 DataFrame으로 다루기 위한 라이브러리임.
- Matplotlib: 그래프를 그려 데이터 분포와 모델 결과를 시각화하는 라이브러리임.
- Seaborn: 통계 그래프를 더 쉽게 그리기 위한 시각화 라이브러리임.


In [1]:
# 초기 세팅용 import 구문입니다. 먼저 실행한 뒤 실습 코드를 작성합니다.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.dates import drange
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.metrics import mean_squared_error, mean_absolute_error, root_mean_squared_error
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet


## California Housing 데이터 불러오기

In [4]:
# 스켈레튼에 켈리포니아 하우징 데이터 불러오기
from sklearn.datasets import fetch_california_housing

california_housing = fetch_california_housing()

# 학습할 입력값(X : X_train, X_test 등등 변경)
california_housing_df = pd.DataFrame(  # 데이터 프레임으로 변경 후
    california_housing.data,  # 데이터 입력
    columns=california_housing.feature_names  # 컬럽값을 지정한다
)

# print(california_housing.target)

# 정답(y) - 예측해야할 주택의 가격 딕셔너리 추가
california_housing_df['MedHouseVal'] = california_housing.target
california_housing_df.head()

,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude,MedHouseVal
0,8.3252,41.0,6.984127,1.023810,322.0,2.555556,37.88,-122.23,4.526
1,8.3014,21.0,6.238137,0.971880,2401.0,2.109842,37.86,-122.22,3.585
2,7.2574,52.0,8.288136,1.073446,496.0,2.802260,37.85,-122.24,3.521
3,5.6431,52.0,5.817352,1.073059,558.0,2.547945,37.85,-122.25,3.413
4,3.8462,52.0,6.281853,1.081081,565.0,2.181467,37.85,-122.25,3.422


## 학습/평가 데이터 분리

- train_test_split: 데이터를 학습용과 평가용으로 나누어 새 데이터 성능을 확인할 준비를 함.


In [5]:
# train_test_split() : 데이터의 일부를 떄서 확인한다
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    california_housing.data,
    california_housing.target,
    test_size=0.2,
    random_state=42
)

print(X_train.shape, y_train.shape)
print(X_test.shape, y_test.shape)

(16512, 8) (16512,)
(4128, 8) (4128,)


## 회귀 평가지표
점수가 낮을수록 좋음
- MSE: 오차를 제곱해 평균낸 값으로 큰 오차에 더 민감함.
- MAE: 오차의 절댓값을 평균낸 값으로 실제 단위 해석이 쉬움.
- RMSE: MSE에 제곱근을 씌워 target과 같은 단위로 해석하는 지표임.


---

점수가 높을수록 좋음
- fit: 훈련 데이터에서 모델 또는 전처리 기준을 학습하는 메서드임.
- predict: 학습된 모델로 새 데이터의 예측값을 생성하는 메서드임.
- score: 모델의 기본 평가 점수를 계산하는 메서드임.


# 다항 feature(다항 선형) - LinearRegression 모델 평가 지표


In [6]:
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.metrics import mean_squared_error, mean_absolute_error, root_mean_squared_error
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression

# 파이프라인이란?
# 모댈생성, 학습, 변환 작업물들을 순서대로 진행하게하는 모델(메소드)
pipeline = Pipeline([
    # 1. 기존 feature를 2차 다항 feature로 확장한다.
    # 예: length, height, width가 있으면
    # length^2, length height, height^2 같은 새로운 feature가 만들어진다.
    # include_bias=False는 상수항 1을 추가하지 않겠다는 의미이다.
    ('poly', PolynomialFeatures(degree=2, include_bias=False)),

    # 2. feature들의 평균과 표준편차를 기준으로 값을 표준화한다.
    # 평균은 0, 표준편차는 1에 가깝게 맞춘다.
    # 다항 feature처럼 값의 크기가 달라질 수 있는 경우 스케일링이 중요하다.
    ('scaler', StandardScaler()),

    # 3. 변환된 feature를 사용해서 선형 회귀 모델을 학습한다.
    # fit_intercept=True는 절편을 학습하겠다는 의미이다.
    ('linear_regression', LinearRegression(fit_intercept=True)),
])

## 학습진행 -> 예측 -> 평가
# 학습 진행
# 파이프라인 전체를 학습한다.
# X_train: 입력 feature, y_train: 정답 target
# 내부적으로 다항 feature 생성 -> 스케일링 -> 선형 회귀 학습 순서로 진행된다.
pipeline.fit(X_train, y_train)

# 학습된 선형 회귀 모델만 따로 확인하고 싶을 때 꺼낸다.
model = pipeline.named_steps['linear_regression']

# 학습 데이터에 대한 예측값
# 파이프라인 내부 변환 과정을 자동으로 거친 뒤 예측한다.
y_train_pred = pipeline.predict(X_train)

# 테스트 데이터에 대한 예측값
# fit은 다시 하지 않고, 학습 때 정한 변환 기준으로 예측한다.
y_test_pred = pipeline.predict(X_test)

# 회귀 모델의 평가 결과를 표 형태로 정리한다.
ridge_eval_result = pd.DataFrame({
    # dataset: 평가 대상 데이터가 train인지 test인지 구분하는 컬럼
    'dataset': ['train', 'test'],

    # R2: 결정계수
    # 모델이 정답 y를 얼마나 잘 설명하는지 나타내는 점수
    # 1에 가까울수록 좋고, 0에 가까우면 평균으로 예측하는 것과 비슷하다.
    'R2': [
        pipeline.score(X_train, y_train),
        pipeline.score(X_test, y_test)
    ],

    # MSE: Mean Squared Error, 평균 제곱 오차
    # 실제값과 예측값의 차이를 제곱한 뒤 평균낸 값
    # 오차를 제곱하기 때문에 큰 오차에 더 민감하다.
    # 작을수록 좋다.
    'MSE': [
        mean_squared_error(y_train, y_train_pred),
        mean_squared_error(y_test, y_test_pred)
    ],

    # MAE: Mean Absolute Error, 평균 절대 오차
    # 실제값과 예측값의 차이를 절댓값으로 바꾼 뒤 평균낸 값
    # 예측값이 실제값과 평균적으로 얼마나 차이 나는지 직관적으로 볼 수 있다.
    # 작을수록 좋다.
    'MAE': [
        mean_absolute_error(y_train, y_train_pred),
        mean_absolute_error(y_test, y_test_pred)
    ],

    # RMSE: Root Mean Squared Error, 평균 제곱근 오차
    # MSE에 루트를 씌운 값
    # target과 같은 단위로 해석할 수 있다.
    # 작을수록 좋다.
    'RMSE': [
        root_mean_squared_error(y_train, y_train_pred),
        root_mean_squared_error(y_test, y_test_pred)
    ]
})

# model.coef_는 학습된 선형 회귀 모델의 회귀계수이다.
# 각 feature가 예측값에 얼마나 영향을 주는지 나타내는 가중치이다.
# 회귀계수 제곱합은 계수들이 전체적으로 얼마나 큰지 확인하는 값이다.
print('회귀계수 제곱합:', np.sum(model.coef_ ** 2))

ridge_eval_result

회귀계수 제곱합: 7170.956218225917


,dataset,R2,MSE,MAE,RMSE
0,train,0.685268,0.420727,0.460838,0.648634
1,test,0.645682,0.464302,0.467001,0.681397


### 다항 feature를 사용한 선형 회귀 모델을 활용한
### Ridge L2 규제 선형 회귀 모델 평가 지표

In [7]:
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.metrics import mean_squared_error, mean_absolute_error, root_mean_squared_error
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge

# 파이프라인이란?
# 모댈생성, 학습, 변환 작업물들을 순서대로 진행하게하는 모델(메소드)
pipeline = Pipeline([
    # 1. 기존 feature를 2차 다항 feature로 확장한다.
    # 예: length, height, width가 있으면
    # length^2, length height, height^2 같은 새로운 feature가 만들어진다.
    # include_bias=False는 상수항 1을 추가하지 않겠다는 의미이다.
    ('poly', PolynomialFeatures(degree=2, include_bias=False)),

    # 2. feature들의 평균과 표준편차를 기준으로 값을 표준화한다.
    # 평균은 0, 표준편차는 1에 가깝게 맞춘다.
    # 다항 feature처럼 값의 크기가 달라질 수 있는 경우 스케일링이 중요하다.
    ('scaler', StandardScaler()),

    # 3. 변환된 feature를 사용해서 선형 회귀 모델을 학습한다.
    # fit_intercept=True는 절편을 학습하겠다는 의미이다.

    # Ridge : 회귀계수 제곱합을 줄이는 규제
    # alpha : 구제 강도

    # 3. 변환된 feature를 사용해서 Ridge 회귀 모델을 학습한다.
    # Ridge는 선형 회귀에 L2 규제를 추가한 모델이다.
    # alpha는 규제의 강도를 조절하는 값이다.
    # alpha가 클수록 회귀계수가 작아지도록 더 강하게 제한한다.
    # alpha=100은 비교적 강한 규제를 적용하겠다는 의미이다.
    # 규제가 강하면 과대적합은 줄어들 수 있지만, 너무 크면 과소적합이 생길 수 있다.
    # 'model'은 파이프라인 안에서 이 Ridge 모델 단계를 부르는 이름이다.
    ('model', Ridge(alpha=1)),
])

## 학습진행 -> 예측 -> 평가
# 학습 진행
# 파이프라인 전체를 학습한다.
# X_train: 입력 feature, y_train: 정답 target
# 내부적으로 다항 feature 생성 -> 스케일링 -> Ridge로 변경
pipeline.fit(X_train, y_train)

# 학습된 선형 회귀 모델만 따로 확인하고 싶을 때 꺼낸다.
model = pipeline.named_steps['model']

# 학습 데이터에 대한 예측값
# 파이프라인 내부 변환 과정을 자동으로 거친 뒤 예측한다.
y_train_pred = pipeline.predict(X_train)

# 테스트 데이터에 대한 예측값
# fit은 다시 하지 않고, 학습 때 정한 변환 기준으로 예측한다.
y_test_pred = pipeline.predict(X_test)

# 회귀 모델의 평가 결과를 표 형태로 정리한다.
ridge_eval_result = pd.DataFrame({
    # dataset: 평가 대상 데이터가 train인지 test인지 구분하는 컬럼
    'dataset': ['train', 'test'],

    # R2: 결정계수
    # 모델이 정답 y를 얼마나 잘 설명하는지 나타내는 점수
    # 1에 가까울수록 좋고, 0에 가까우면 평균으로 예측하는 것과 비슷하다.
    'R2': [
        pipeline.score(X_train, y_train),
        pipeline.score(X_test, y_test)
    ],

    # MSE: Mean Squared Error, 평균 제곱 오차
    # 실제값과 예측값의 차이를 제곱한 뒤 평균낸 값
    # 오차를 제곱하기 때문에 큰 오차에 더 민감하다.
    # 작을수록 좋다.
    'MSE': [
        mean_squared_error(y_train, y_train_pred),
        mean_squared_error(y_test, y_test_pred)
    ],

    # MAE: Mean Absolute Error, 평균 절대 오차
    # 실제값과 예측값의 차이를 절댓값으로 바꾼 뒤 평균낸 값
    # 예측값이 실제값과 평균적으로 얼마나 차이 나는지 직관적으로 볼 수 있다.
    # 작을수록 좋다.
    'MAE': [
        mean_absolute_error(y_train, y_train_pred),
        mean_absolute_error(y_test, y_test_pred)
    ],

    # RMSE: Root Mean Squared Error, 평균 제곱근 오차
    # MSE에 루트를 씌운 값
    # target과 같은 단위로 해석할 수 있다.
    # 작을수록 좋다.
    'RMSE': [
        root_mean_squared_error(y_train, y_train_pred),
        root_mean_squared_error(y_test, y_test_pred)
    ]
})

# model.coef_는 학습된 선형 회귀 모델의 회귀계수이다.
# 각 feature가 예측값에 얼마나 영향을 주는지 나타내는 가중치이다.
# 회귀계수 제곱합은 계수들이 전체적으로 얼마나 큰지 확인하는 값이다.
print('회귀계수 제곱합:', np.sum(model.coef_ ** 2))

ridge_eval_result

# L2(Ridge) 규제는 다항 feature의 표현력을 높이면서 alpha로 회귀계수를 제한
# alpha값에 따라 R2, MSE 값들이 달라진다


회귀계수 제곱합: 119.22218689328248


,dataset,R2,MSE,MAE,RMSE
0,train,0.669386,0.441958,0.482729,0.664799
1,test,0.639133,0.472883,0.487971,0.687665


### 최적의 alpha값 찾기


## 교차검증

- cross_val_score: 여러 fold의 검증 점수를 계산해 평균 성능을 더 안정적으로 확인함.
- cross_val_score() : 학습데이터를 여러 fold(데이터들을 접어서 구간을 나눔)로 나눠 검증 점수를 계산



In [8]:
from sklearn.model_selection import cross_val_score

# cross_val_score() : 학습데이터를 여러 fold(데이터들을 접어서 구간을 나눔)로 나눠 검증 점수를 계산

# 테스트해볼 alpha값 준비
alphas = [0, 0.1, 1, 10, 200, 300]
# 검증좀수를 모아둘 리스트
# 각 alpha 값마다 교차 검증으로 나온 평균 MSE를 저장할 리스트이다.
cv_results = []

# alpha 값을 하나씩 바꿔가며 반복한다.
for alpha in alphas:
    # 현재 alpha 값을 사용하는 파이프라인을 새로 만든다.
    # alpha마다 Ridge 모델이 달라지므로 반복문 안에서 pipeline을 다시 생성한다.
    pipeline = Pipeline([
        # 1. 기존 feature를 2차 다항 feature로 확장한다.
        ('poly', PolynomialFeatures(degree=2, include_bias=False)),

        # 2. 확장된 feature들의 단위를 평균 0, 표준편차 1 기준으로 맞춘다.
        ('scaler', StandardScaler()),

        # 3. 현재 alpha 값을 사용하는 Ridge 회귀 모델을 만든다.
        ('model', Ridge(alpha=alpha))
    ])

    # cross_val_score는 cv=5이므로 학습 데이터를 5등분한다.
    # 그중 4개 조각으로 학습하고, 나머지 1개 조각으로 검증한다.
    # 이 과정을 검증 조각을 바꿔가며 총 5번 반복한다.

    # scoring='neg_mean_squared_error'
    # sklearn은 점수가 클수록 좋은 방향으로 통일하기 위해
    # MSE에 음수(-)를 붙인 값을 반환한다.
    # 하지만 우리가 해석할 때는 양수 MSE가 편하므로 -1을 곱해서 다시 양수로 바꾼다.
    scores = -1 * cross_val_score(
        pipeline,  # 평가할 모델 파이프라인
        X_train,  # 교차 검증에 사용할 입력 feature
        y_train,  # 교차 검증에 사용할 정답 target
        cv=5,  # 데이터를 5개 fold로 나누어 검증(5개의 데이터 조각)
        scoring='neg_mean_squared_error'  # 데이터를 나누어 음수로 표시한다 # 평가 기준: MSE
    )

    # 현재 alpha에서 나온 5번의 MSE 평균을 저장한다.
    # 평균 MSE가 작을수록 더 좋은 alpha라고 볼 수 있다.
    cv_results.append({
        'alpha': alpha,  # 알파값은 얼마고
        'mean_MSE': scores.mean(),  # 평균은 얼마고
        'std_MSE': scores.std()  # 표준편차는 얼마인가
    })

ridge_cv_results = pd.DataFrame(cv_results)
ridge_cv_results



,alpha,mean_MSE,std_MSE
0,0.0,10.448255,18.601131
1,0.1,1.812390,2.335427
2,1.0,1.005593,0.782599
3,10.0,4.019982,6.748552
4,200.0,1.658760,2.318480
5,300.0,1.249920,1.498657


## L1
- L1규제방식을 구현한 Lasso클래스를 사용할 수 있다.
- alpha값을 통해 특정 회귀계수를 0까지 제한, 특정속성을 회귀계산에서 배제하는 것도 가능.


In [9]:
# alpha값이 커질수록 강햔 규제가 적용되어 feature가 많이 사라짐
from sklearn.linear_model import Lasso

# cross_val_score() : 학습데이터를 여러 fold(데이터들을 접어서 구간을 나눔)로 나눠 검증 점수를 계산


# Lasso는 선형 회귀에 L1 규제를 추가한 모델이다.
# L1 규제는 중요하지 않은 feature의 회귀계수를 0으로 만들 수 있다.
# 그래서 Lasso는 feature 선택 효과가 있다.

# alpha값이 커질수록 강한 규제가 적용된다.
# 규제가 강해지면 더 많은 feature의 계수가 0이 될 수 있다.
alphas = [0.001, 0.003, 0.07, 0.1, 0.5, 1]

# alpha별 feature 계수를 저장할 표이다.
# 각 열에는 특정 alpha에서 학습된 회귀계수가 들어간다.
coef_df = pd.DataFrame()

# alpha별 평가 결과를 저장할 리스트이다.
lass_results = []

for alpha in alphas:
    # alpha 값을 하나씩 바꿔가며 Lasso 모델을 만든다.
    pipeline = Pipeline([
        # 기존 feature를 2차 다항 feature로 확장한다.
        ('poly', PolynomialFeatures(degree=2, include_bias=False)),

        # feature들의 단위를 평균 0, 표준편차 1 기준으로 맞춘다.
        ('scaler', StandardScaler()),

        # Lasso 회귀 모델을 만든다.
        # alpha는 규제 강도이다.
        # max_iter는 반복 학습 횟수를 늘려 수렴 경고를 줄이기 위해 사용한다.
        # ('lasso', Lasso(alpha=alpha, max_iter=10000)),
        ('model', Lasso(alpha=alpha, max_iter=200000)),
    ])

    # 학습 데이터로 파이프라인 전체를 학습한다.
    # 실행 순서: 다항 feature 생성 -> 스케일링 -> Lasso 학습
    pipeline.fit(X_train, y_train)

    # 파이프라인 안에서 학습된 Lasso 모델을 꺼낸다.
    # 위에서 단계 이름을 'lasso'로 만들었기 때문에 같은 이름으로 접근한다.
    lasso_model = pipeline.named_steps['model']

    # 다항 변환 후 만들어진 feature 이름을 가져온다.
    # X_train이 DataFrame이면 컬럼명을 사용하고,
    # numpy array이면 x0, x1, x2 같은 기본 이름을 사용한다.
    if hasattr(X_train, "columns"):
        feature = pipeline.named_steps['poly'].get_feature_names_out(X_train.columns)
    else:
        feature = pipeline.named_steps['poly'].get_feature_names_out()

    # 각 alpha에서 학습된 feature별 회귀계수를 저장한다.
    # 계수가 0이면 해당 feature는 Lasso에 의해 거의 사용되지 않는다고 볼 수 있다.
    coef_df[f'alpha_{alpha}'] = pd.Series(
        lasso_model.coef_,
        index=feature
    )

    # 테스트 데이터에 대한 예측값을 만든다.
    y_test_pred = pipeline.predict(X_test)

    # 현재 alpha의 평가 결과를 저장한다.
    lass_results.append({
        # 현재 사용한 alpha 값
        'alpha': alpha,

        # MSE: 실제값과 예측값의 차이를 제곱해서 평균낸 값
        # 작을수록 좋다.
        'MSE': mean_squared_error(y_test, y_test_pred),

        # R2: 모델이 정답을 얼마나 잘 설명하는지 나타내는 점수
        # 1에 가까울수록 좋다.
        'R2': pipeline.score(X_test, y_test),

        # 회귀계수가 0에 가까운 feature 개수
        # 이 값이 클수록 Lasso가 많은 feature를 제거한 것이다.
        'zero_coef_count': np.sum(np.isclose(lasso_model.coef_, 0)),
    })

# 리스트로 모은 평가 결과를 표로 변환한다.
lass_result_df = pd.DataFrame(lass_results)
lass_result_df

# Lasso는 alpha가 커질수록 규제가 강해져 0이되는 계수의 수가 증가한다
# -> feature가 많이 제거되어 예측 성능이 하락할수있따

# w0x0, w1x1, w2x2에서 헷갈리는 지점은 회귀계수는 w이고, x는 feature 값이라는 점입니다.
# 선형 회귀식은 보통 이렇게 봅니다.
#
# 예측값 = w0*x0 + w1*x1 + w2*x2 + b
#
# 각각의 의미는:
#
# x0, x1, x2 = feature 값
# w0, w1, w2 = 각 feature에 곱해지는 회귀계수, 가중치
# b          = 절편, intercept
#
# 예를 들어 농어 무게를 예측한다고 하면:
#
# x0 = length
# x1 = height
# x2 = width
#
# 모델이 학습해서 이런 계수를 얻었다고 해보면:
#
# w0 = 30
# w1 = 80
# w2 = 120
# b  = -500
#
# 그러면 예측식은 이렇게 됩니다.
#
# 예측 무게 = 30*length + 80*height + 120*width - 500
#
# 즉:
#
# 30*length  = 길이가 예측값에 기여하는 부분
# 80*height  = 높이가 예측값에 기여하는 부분
# 120*width  = 너비가 예측값에 기여하는 부분
#
# 그래서 w0x0 전체가 회귀계수는 아닙니다.
#
# 정확히는:
#
# w0 = 회귀계수
# x0 = feature 값
# w0*x0 = 그 feature가 예측값에 기여한 값
#
# 예를 들어 length=20, height=5, width=3이면:
#
# 예측값 = 30*20 + 80*5 + 120*3 - 500
#      = 600 + 400 + 360 - 500
#      = 860
#
# 여기서 모델이 학습한 것은 w0, w1, w2, b입니다.
# 우리가 입력하는 값은 x0, x1, x2입니다.
#
# 정리하면:
#
# feature       = x
# 회귀계수      = w
# 절편          = b
# 예측값        = w*x 들을 모두 더한 값 + b
#
# model.coef_를 출력하면 나오는 값들이 바로 w0, w1, w2 같은 회귀계수입니다.
# model.intercept_를 출력하면 b, 즉 절편이 나옵니다.


,alpha,MSE,R2,zero_coef_count
0,0.001,0.483431,0.631084,17
1,0.003,0.505882,0.613951,20
2,0.070,0.659940,0.496386,41
3,0.100,0.669386,0.489178,41
4,0.500,0.937914,0.284259,43
5,1.000,1.310696,-0.000219,44


## 회귀 평가지표

- MSE: 오차를 제곱해 평균낸 값으로 큰 오차에 더 민감함.
- fit: 훈련 데이터에서 모델 또는 전처리 기준을 학습하는 메서드임.
- predict: 학습된 모델로 새 데이터의 예측값을 생성하는 메서드임.


## ElasticNet L1 + L2
L1규제, L2규제를 적절한 비율로 모두 적용하는 ElasticNet 선형회귀모델을 사용할 수 있다.
ElasticNet은 **회귀 분석** 기법 중 하나로, **Lasso**와 **Ridge**의 규제를 결합한 모델이다.


Lasso는 특성 선택에 효과적이고, Ridge는 모든 특성을 다루면서 모델을 규제한다.
ElasticNet은 이 두 가지 규제(L1과 L2)를 적절히 혼합하여 사용하는 방법이다.

**alpha 파라미터**
alpha는 a + b를 의미한다.
- a는 L1규제용 alpha값이다.
- b는 L2규제용 alpha값이다.

**l1_ratio 파라미터**
L1규제용 alpha값의 비율이다. $\frac{a}{a + b}$
- alpha가 10이고, l1_ratio가 0.7이면 a = 7, b = 3이다.
- alpha가 10이고, l1_ratio가 1이면 a = 10, b = 0이다. 즉, L1규제만 사용한다.
- alpha가 10이고, l1_ratio가 0이면 a = 0, b = 10이다. 즉, L2규제만 사용한다.

**수식:**

$$J(β) = RSS + α [ λ * ||β||₁ + (1 - λ) * ||β||₂² ]$$
- **RSS**: Residual Sum of Squares (예측 오차)
- **α**: 전체 규제 강도 (크면 규제가 강해짐)
- **λ**: L1과 L2 규제의 비율 조절 (0 ≤ λ ≤ 1)
  - λ = 1 → Lasso만 적용
  - λ = 0 → Ridge만 적용
- **||β||₁**: L1 노름 (∑|βᵢ|), 특성 선택
- **||β||₂²**: L2 노름 제곱 (∑βᵢ²), 계수 축소

**특징:**
- **Lasso와 Ridge의 장점을 결합**: ElasticNet은 Lasso의 **특성 선택** 능력과 Ridge의 **강한 규제** 특성을 모두 반영한다.
- **고차원 데이터에 적합**: 상관관계가 높은 특성이 많은 데이터나 차원이 높은 데이터에 적합하다.
- **Overfitting 방지**: 두 가지 규제를 혼합하여 과적합을 효과적으로 방지할 수 있다.


## 모델 학습

- fit: 훈련 데이터에서 모델 또는 전처리 기준을 학습하는 메서드임.
- score: 모델의 기본 평가 점수를 계산하는 메서드임.


In [12]:
from sklearn.linear_model import ElasticNet

# 전처리와 모델을 하나로 묶는 파이프라인 생성
pipeline = Pipeline([
    # 입력 특성을 2차 다항 특성으로 확장
    # 예: x1, x2 -> x1, x2, x1^2, x1*x2, x2^2
    ('poly', PolynomialFeatures(degree=2, include_bias=False)),

    # 각 특성의 스케일을 평균 0, 표준편차 1로 표준화
    # 규제 모델은 특성 스케일의 영향을 많이 받으므로 StandardScaler를 사용
    ('scaler', StandardScaler()),

    # ElasticNet : L1 +L2 규제를 함께 사용하는 모델
    # alpha : 전체 규제 강도
    # l1_ratio : L1의 규제 비율, 0에 가까울수록 Ridge, 1에 가까울수록 Lassod와 가깝다
    # max_iter: 회귀계수의 최적값을 찾기 위한 반복 횟수
    ('model', ElasticNet(alpha=0.1, l1_ratio=0.1, max_iter=20000))
])

# 학습용 데이터를 fix에 학습 , train이 학습
# X_train과 y_train을 사용해 파이프라인 전체를 학습
# poly -> scaler -> model 순서로 실행됨
pipeline.fit(X_train, y_train)

# 학습 데이터와 테스트 데이터의 R2 점수를 DataFrame으로 정리
elasticnet_result = pd.DataFrame({
    # 평가할 데이터셋 이름
    'dataset': ['train', 'test'],

    'R2': [  # score
        # 학습 데이터에 대한 R2 점수
        pipeline.score(X_train, y_train),

        # 테스트 데이터에 대한 R2 점수
        pipeline.score(X_test, y_test)
    ]
})


## 결과 확인

- 실행 결과: 앞에서 만든 객체와 실행 결과를 확인하며 다음 단계로 연결함.


In [11]:
# ElasticNet 모델의 train/test 성능 결과 출력
elasticnet_result

,dataset,R2
0,train,0.592978
1,test,0.550920


## 다중공선성 MultiCollinearity
특성간의 상관관계가 너무 높은 경우를 가리킨다.
주택데이터에서 면적특성과 방의크기특성은 높은 상관관계(상관계수 0.8이상)를 가질수 있다.
다중공선성특성에 대한 회귀계수가 크게 학습이 되고, 이는 특정데이터에 민감한 과대적합을 유발한다.
**해결책**
- 다중공선성 특성 제거
- 규제모델을 사용한 회귀계수 억제


## 샘플 데이터 생성

In [13]:
np.random.seed(42)

# 첫 번째 feature X_corr를 만들고, 두 번째/세 번째 feature를 X_corr의 제곱/세제곱 기반으로 만듦.
# 이렇게 만들면 feature들이 서로 강하게 연관되어 다중공선성이 생김.
X_corr = np.random.rand(100, 1)
X_corr = np.column_stack((X_corr, X_corr ** 2 + 3, X_corr ** 3 + 4))

# target은 예제용 난수로 생성함.
y_corr = 3 * np.random.randn(100, 1) + np.random.randn(100, 1)
y_corr = y_corr.ravel()

# np.corrcoef(): 변수 간 상관계수를 계산함.
# rowvar=False: 행이 아니라 컬럼을 변수(feature)로 보고 상관계수를 계산함.
corr_mat = np.corrcoef(X_corr, rowvar=False)
corr_mat

array([[1.        , 0.96876011, 0.91658615],
       [0.96876011, 1.        , 0.98569849],
       [0.91658615, 0.98569849, 1.        ]])

## 학습/평가 데이터 분리

- train_test_split: 데이터를 학습용과 평가용으로 나누어 새 데이터 성능을 확인할 준비를 함.
- MSE: 오차를 제곱해 평균낸 값으로 큰 오차에 더 민감함.
- fit: 훈련 데이터에서 모델 또는 전처리 기준을 학습하는 메서드임.
- predict: 학습된 모델로 새 데이터의 예측값을 생성하는 메서드임.


In [14]:
 # train_test_split(): 샘플 데이터를 학습용과 평가용으로 나눔.
# X_corr: 모델이 보고 학습할 입력 데이터(feature)
# y_corr: 모델이 맞혀야 하는 정답 데이터(target)
# test_size=0.2: 전체 데이터 중 20%는 평가용(test), 80%는 학습용(train)으로 사용
# random_state=42: 실행할 때마다 같은 방식으로 데이터가 나뉘도록 고정
X_corr_train, X_corr_test, y_corr_train, y_corr_test = train_test_split(
    X_corr,
    y_corr,
    test_size=0.2,
    random_state=42
)

# 선형 회귀 모델들을 불러옴.
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet

# 비교할 모델 목록임.
# LinearRegression: 규제 없음
# Ridge: L2 규제
# Lasso: L1 규제
# ElasticNet: L1 + L2 규제
models = {
    # 기본 선형 회귀 모델
    # 규제를 사용하지 않기 때문에 계수 값이 커질 수 있음.
    'LinearRegression': LinearRegression(),

    # Ridge 회귀 모델
    # alpha=10.0: L2 규제 강도
    # alpha 값이 클수록 회귀계수를 더 작게 만들려고 함.
    'Ridge': Ridge(alpha=10.0),

    # Lasso 회귀 모델
    # alpha=0.1: L1 규제 강도
    # max_iter=5000: 최적의 회귀계수를 찾기 위한 최대 반복 횟수
    # Lasso는 중요하지 않은 feature의 계수를 0으로 만들 수 있음.
    'Lasso': Lasso(alpha=0.1, max_iter=5000),

    # ElasticNet 회귀 모델
    # alpha=0.1: 전체 규제 강도
    # l1_ratio=0.5: L1 규제와 L2 규제를 반반 섞어서 사용
    # max_iter=5000: 최적의 회귀계수를 찾기 위한 최대 반복 횟수
    'ElasticNet': ElasticNet(alpha=0.1, l1_ratio=0.5, max_iter=5000),
}

# 각 모델의 평가 결과를 저장할 빈 리스트
corr_results = []

# models 딕셔너리에서 모델 이름(name)과 모델 객체(model)를 하나씩 꺼냄.
for name, model in models.items():
    # 전처리와 모델 학습을 하나로 묶는 Pipeline 생성
    # Pipeline을 사용하면 scaler 적용 후 model 학습까지 순서대로 실행됨.
    pipeline = Pipeline([
        # StandardScaler(): feature 스케일을 맞춰 규제 모델의 계수 벌점이 공정하게 적용되도록 함.
        # 스케일링: feature들의 숫자 범위를 비슷하게 맞추는 과정
        # 예를 들어 어떤 feature는 1~10, 다른 feature는 1000~10000이면
        # 숫자 범위가 큰 feature가 모델에 더 큰 영향을 줄 수 있음.
        # StandardScaler는 각 feature를 평균 0, 표준편차 1 기준으로 변환함.
        ('scaler', StandardScaler()),

        # 위에서 꺼낸 모델을 pipeline의 마지막 단계로 넣음.
        # scaler로 변환된 데이터를 사용해 이 모델이 학습됨.
        ('model', model)
    ])

    # 학습 데이터로 pipeline 전체를 학습함.
    # 실행 순서: StandardScaler가 X_corr_train 기준으로 스케일링 방법 학습
    #          -> X_corr_train을 스케일링
    #          -> model이 스케일링된 X_corr_train과 y_corr_train으로 학습
    pipeline.fit(X_corr_train, y_corr_train)

    # pipeline 안에서 학습이 끝난 실제 모델만 꺼냄.
    # 회귀계수(coef_)를 확인하기 위해 사용함.
    trained_model = pipeline.named_steps['model']

    # 현재 모델의 평가 결과를 딕셔너리 형태로 저장
    corr_results.append({
        # 모델 이름
        'model': name,

        # 학습 데이터의 MSE
        # MSE는 예측값과 실제값의 차이를 제곱해서 평균낸 값
        # 값이 작을수록 예측 오차가 작다는 뜻
        'train_MSE': mean_squared_error(y_corr_train, pipeline.predict(X_corr_train)),

        # 평가 데이터의 MSE
        # 모델이 처음 보는 데이터에서 얼마나 잘 예측하는지 확인
        'test_MSE': mean_squared_error(y_corr_test, pipeline.predict(X_corr_test)),

        # 회귀계수들을 제곱해서 모두 더한 값
        # 값이 클수록 모델의 계수 크기가 크다는 의미
        # Ridge 같은 L2 규제는 이 값을 줄이는 방향으로 학습함.
        'coef_squared_sum': np.sum(trained_model.coef_ ** 2),

        # 계수가 0에 가까운 feature 개수
        # Lasso나 ElasticNet은 중요도가 낮은 feature의 계수를 0으로 만들 수 있음.
        'zero_coef_count': np.sum(np.isclose(trained_model.coef_, 0))
    })

# 리스트에 저장된 모델별 결과를 DataFrame으로 변환해서 표 형태로 출력
pd.DataFrame(corr_results)


# 의도한 내용은
# 회귀계수 또는 feature 개수를 규제해서 과적합 해결이 가능함

,model,train_MSE,test_MSE,coef_squared_sum,zero_coef_count
0,LinearRegression,7.217016,10.176685,97.534503,0
1,Ridge,7.508614,10.527044,1.091929,0
2,Lasso,7.554207,10.601655,0.784001,1
3,ElasticNet,7.502225,10.505844,1.162947,1


In [15]:
# 랜덤값을 고정함.
# random seed를 고정하면 코드를 다시 실행해도 같은 랜덤 데이터가 만들어짐.
# 그래서 실습 결과를 매번 똑같이 확인할 수 있음.
np.random.seed(42)

# 만들 데이터 개수
# 즉, 샘플 200개를 만든다는 뜻
n = 200

# x1은 핵심 feature
# 평균 0, 표준편차 1을 따르는 랜덤 숫자 200개를 생성
# 이 예제에서 x1은 target(y_corr)에 실제로 영향을 주는 중요한 feature임.
x1 = np.random.normal(0, 1, n)

# x2, x3는 x1과 거의 같은 feature
# 작은 noise만 더해서 거의 중복된 컬럼처럼 만듦.
# x1 값에 아주 작은 랜덤값을 더해서 x2를 만듦.
# 그래서 x2는 x1과 거의 같은 움직임을 보임.
x2 = x1 + np.random.normal(0, 0.01, n)

# x3도 x1 값에 아주 작은 랜덤값을 더해서 만듦.
# x1, x2, x3는 서로 매우 강하게 묶인 feature가 됨.
# 이런 상태를 feature 간 상관관계가 높다고 말함.
x3 = x1 + np.random.normal(0, 0.01, n)

# x4는 정답과 거의 관계없는 독립 feature
# x1과 상관없이 따로 랜덤하게 만들어진 feature
# target을 만들 때 x4는 사용하지 않으므로, 예측에 크게 도움이 되지 않는 noise feature임.
x4 = np.random.normal(0, 1, n)

# x1, x2, x3, x4를 하나의 2차원 배열로 합침.
# 모델은 보통 여러 feature를 하나의 X 데이터로 받아서 학습함.
# 결과 shape는 (200, 4)가 됨. 즉, 200행 4열 데이터
X_corr = np.column_stack([x1, x2, x3, x4])

# 각 feature의 이름을 지정함.
# 나중에 DataFrame으로 만들거나 상관관계를 볼 때 컬럼 이름으로 사용함.
feature_names = ['x1', 'x2_almost_x1', 'x3_almost_x1', 'x4_noise']

# target은 실제로 x1의 영향을 받도록 만듦.
# 이렇게 해야 모델이 학습할 신호가 생김.
# y_corr = 5 * x1 + 약간의 랜덤 오차
# 즉, x1이 커지면 y_corr도 대체로 커지는 구조임.
# np.random.normal(0, 0.5, n)은 현실 데이터처럼 약간의 오차를 넣기 위한 noise임.
y_corr = 5 * x1 + np.random.normal(0, 0.5, n)

# X_corr 배열을 DataFrame으로 바꾸고 컬럼 이름을 붙임.
# corr()은 컬럼들끼리 얼마나 비슷하게 움직이는지 상관계수를 계산함.
# 상관계수는 -1부터 1까지의 값을 가짐.
# 1에 가까우면 같이 증가하는 관계가 강함.
# -1에 가까우면 한쪽이 증가할 때 다른 쪽은 감소하는 관계가 강함.
# 0에 가까우면 관계가 거의 없음.
corr_df = pd.DataFrame(X_corr, columns=feature_names).corr()

# feature 간 상관관계 표 출력
# 여기서는 x1, x2_almost_x1, x3_almost_x1의 상관관계가 1에 매우 가깝게 나올 것임.
# 즉, 세 feature가 거의 같은 정보를 담고 있다는 뜻
corr_df

,x1,x2_almost_x1,x3_almost_x1,x4_noise
x1,1.000000,0.999944,0.999944,0.065398
x2_almost_x1,0.999944,1.000000,0.999886,0.064254
x3_almost_x1,0.999944,0.999886,1.000000,0.066621
x4_noise,0.065398,0.064254,0.066621,1.000000


In [16]:
# 데이터를 학습용(train)과 평가용(test)으로 나누기 위한 함수
from sklearn.model_selection import train_test_split

# 비교할 회귀 모델들
# LinearRegression: 규제 없는 기본 선형 회귀
# Ridge: L2 규제를 사용하는 회귀
# Lasso: L1 규제를 사용하는 회귀
# ElasticNet: L1 + L2 규제를 함께 사용하는 회귀
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet

# 전처리와 모델을 하나로 묶어서 순서대로 실행하게 해주는 도구
from sklearn.pipeline import Pipeline

# feature들의 숫자 범위를 비슷하게 맞추는 스케일링 도구
from sklearn.preprocessing import StandardScaler

# 모델 평가 지표
# mean_squared_error: MSE 계산
# r2_score: R2 점수 계산
from sklearn.metrics import mean_squared_error, r2_score

# X_corr, y_corr 데이터를 학습용과 평가용으로 나눔.
# X_corr: 모델이 보고 학습할 입력 데이터
# y_corr: 모델이 맞혀야 하는 정답 데이터
# test_size=0.2: 전체 데이터 중 20%를 평가용으로 사용
# random_state=42: 실행할 때마다 같은 방식으로 데이터가 나뉘도록 고정
X_corr_train, X_corr_test, y_corr_train, y_corr_test = train_test_split(
    X_corr,
    y_corr,
    test_size=0.2,
    random_state=42
)

# 비교할 모델들을 딕셔너리로 정리
# key는 모델 이름, value는 실제 모델 객체
models = {
    # 규제가 없는 기본 선형 회귀 모델
    'LinearRegression': LinearRegression(),

    # Ridge 회귀 모델
    # alpha=10.0: L2 규제 강도
    # alpha가 클수록 회귀계수를 작게 만들려고 함.
    'Ridge': Ridge(alpha=10.0),

    # Lasso 회귀 모델
    # alpha=0.05: L1 규제 강도
    # Lasso는 중요하지 않은 feature의 계수를 0으로 만들 수 있음.
    # max_iter=10000: 최적의 계수를 찾기 위한 최대 반복 횟수
    'Lasso': Lasso(alpha=0.05, max_iter=10000),

    # ElasticNet 회귀 모델
    # alpha=0.05: 전체 규제 강도
    # l1_ratio=0.5: L1 규제와 L2 규제를 반반 섞어서 사용
    # max_iter=10000: 최적의 계수를 찾기 위한 최대 반복 횟수
    'ElasticNet': ElasticNet(alpha=0.05, l1_ratio=0.5, max_iter=10000),
}

# 모델별 평가 결과를 저장할 빈 리스트
results = []

# 모델별 회귀계수를 저장할 DataFrame
# index=feature_names로 설정해서 각 행이 feature 이름이 되도록 함.
coef_df = pd.DataFrame(index=feature_names)

# models 딕셔너리에서 모델 이름(name)과 모델 객체(model)를 하나씩 꺼내 반복
for name, model in models.items():
    # 스케일링과 모델 학습을 하나로 묶은 Pipeline 생성
    pipeline = Pipeline([
        # StandardScaler(): feature들의 스케일을 평균 0, 표준편차 1로 맞춤.
        # 규제 모델(Ridge, Lasso, ElasticNet)은 계수 크기에 벌점을 주기 때문에
        # feature 스케일이 다르면 벌점이 불공평하게 적용될 수 있음.
        # 그래서 모델 학습 전에 스케일링을 먼저 해줌.
        ('scaler', StandardScaler()),

        # 현재 반복 중인 모델을 pipeline에 넣음.
        # scaler가 변환한 데이터를 이 모델이 학습함.
        ('model', model)
    ])

    # 학습 데이터로 pipeline 전체를 학습함.
    # 실행 순서:
    # 1. X_corr_train을 기준으로 스케일링 기준을 학습
    # 2. X_corr_train을 스케일링
    # 3. 스케일링된 X_corr_train과 y_corr_train으로 모델 학습
    pipeline.fit(X_corr_train, y_corr_train)

    # 학습된 pipeline으로 평가용 데이터 X_corr_test를 예측함.
    # Pipeline 안의 scaler가 X_corr_test도 같은 기준으로 스케일링한 뒤 모델이 예측함.
    pred = pipeline.predict(X_corr_test)

    # pipeline 안에서 학습이 끝난 실제 모델만 꺼냄.
    # 회귀계수(coef_)를 확인하기 위해 사용함.
    trained_model = pipeline.named_steps['model']

    # 현재 모델의 회귀계수를 coef_df에 저장
    # 각 feature가 예측에 얼마나 영향을 주는지 확인할 수 있음.
    coef_df[name] = trained_model.coef_

    # 현재 모델의 평가 결과를 딕셔너리로 저장
    results.append({
        # 모델 이름
        'model': name,

        # 테스트 데이터 R2 점수
        # 1에 가까울수록 모델이 정답을 잘 설명한다는 뜻
        'test_R2': r2_score(y_corr_test, pred),

        # 테스트 데이터 MSE
        # 예측값과 실제값의 차이를 제곱해서 평균낸 값
        # 값이 작을수록 예측 오차가 작음.
        'test_MSE': mean_squared_error(y_corr_test, pred),

        # 회귀계수 절댓값의 합
        # 모델이 전체적으로 얼마나 큰 계수를 사용했는지 보는 값
        # 규제가 강하면 보통 이 값이 작아짐.
        'coef_abs_sum': np.sum(np.abs(trained_model.coef_)),

        # 계수가 0에 가까운 feature 개수
        # Lasso나 ElasticNet은 필요 없는 feature의 계수를 0으로 만들 수 있음.
        'zero_coef_count': np.sum(np.isclose(trained_model.coef_, 0))
    })

# 모델별 평가 결과를 표로 출력
display(pd.DataFrame(results))

# 모델별 feature 회귀계수를 표로 출력
# 어떤 모델이 어떤 feature에 큰 계수를 주었는지 비교할 수 있음.
display(coef_df)

,model,test_R2,test_MSE,coef_abs_sum,zero_coef_count
0,LinearRegression,0.986293,0.258187,18.374808,0
1,Ridge,0.986690,0.250704,4.693612,0
2,Lasso,0.986987,0.245118,4.694631,1
3,ElasticNet,0.986773,0.249136,4.704208,0


,LinearRegression,Ridge,Lasso,ElasticNet
x1,9.968837,1.536908,4.645945,1.552183
x2_almost_x1,-6.798938,1.523699,0.000000,1.519741
x3_almost_x1,1.524100,1.536945,0.000686,1.559146
x4_noise,0.082932,0.096060,0.048000,0.073138
